# Small Language Models Training Pipeline
## Phi, Qwen for MTV Camera COQE Prefix Task
This notebook refactors the training pipeline to support **causal language models** (Phi, Qwen) instead of seq2seq models (T5).

# Setup

In [ ]:
!pip install transformers -q
!pip install datasets -q
!pip install rouge -q
!pip install torch -q
!pip install tqdm -q

In [ ]:
import json
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import os
import string
import operator
import random

from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW  # Updated: AdamW now in torch.optim
from transformers import AutoTokenizer, AutoModelForCausalLM  # Changed from AutoModelForSeq2SeqLM
from transformers import get_linear_schedule_with_warmup
from tqdm import tqdm, trange

seed = 42
torch.cuda.empty_cache()
device = torch.device('cuda')
print(f"Device: {device}")

# Configuration

In [ ]:
# ========== MODEL SELECTION ==========
# Small LLM models (< 1B parameters) to compare
AVAILABLE_MODELS = {
    'phi-2': 'microsoft/phi-2',
    'phi-1': 'microsoft/phi-1',
    'qwen-0.5b': 'Qwen/Qwen2-0.5B',
    'qwen-1b': 'Qwen/Qwen2-1B',
    'tinyllama': 'TinyLlama/TinyLlama-1.1B-Chat-v1.0',
}

# Select model to train
MODEL_NAME = 'qwen-0.5b'  # Change this to test different models
model_checkpoint = AVAILABLE_MODELS[MODEL_NAME]

print(f"\n{'='*50}")
print(f"Selected Model: {MODEL_NAME}")
print(f"Model Checkpoint: {model_checkpoint}")
print(f"{'='*50}\n")

# ========== TRAINING PARAMETERS ==========
n_gpu = '0,1'  # Sử dụng cả 2 GPU
train_batch_size = 8  # Tăng batch size để tận dụng bộ nhớ GPU
eval_batch_size = 16  # Batch size lớn hơn cho đánh giá
gradient_accumulation_steps = 4  # Tăng tích lũy gradient để giảm bộ nhớ
lr = 5e-4
adam_epsilon = 1e-8
weight_decay = 0.000001
num_warmup_steps = 0.0
num_train_epochs = 1
save_model = True
save_last_k = 1
max_seq_length = 256

# ========== MEMORY OPTIMIZATION ==========
use_gradient_checkpointing = True  # Giảm bộ nhớ sử dụng
use_mixed_precision = True  # Sử dụng mixed precision (fp16) để tăng tốc độ

elem_dict = ["subject", "object", "aspect", "predicate", "label"]

# ========== DATA PATHS ==========
# Updated dataset path
data_dir = "/kaggle/input/datasets/lynss462/t5-camera-coqe-data"
working_dir = "/kaggle/working"
result_dir = f"{working_dir}/result/model-{MODEL_NAME}"
inference_dir = f"{working_dir}/result/inference-{MODEL_NAME}"

if not os.path.exists(result_dir):
    os.makedirs(result_dir, exist_ok=True)
if not os.path.exists(inference_dir):
    os.makedirs(inference_dir, exist_ok=True)

print(f"Data dir: {data_dir}")
print(f"Result dir: {result_dir}")
print(f"Inference dir: {inference_dir}")

print(f"\n⚙️  Memory Optimization Settings:")
print(f"  Batch Size: {train_batch_size}")
print(f"  Gradient Accumulation Steps: {gradient_accumulation_steps}")
print(f"  Gradient Checkpointing: {use_gradient_checkpointing}")
print(f"  Mixed Precision: {use_mixed_precision}")

# Verify data files exist
import os
train_path = os.path.join(data_dir, 'train.txt')
dev_path = os.path.join(data_dir, 'dev.txt')
test_path = os.path.join(data_dir, 'test.txt')

print(f"\n✓ Dataset files found:")
print(f"  train.txt: {'✓' if os.path.exists(train_path) else '✗'} ({os.path.getsize(train_path) if os.path.exists(train_path) else 0} bytes)")
print(f"  dev.txt:   {'✓' if os.path.exists(dev_path) else '✗'} ({os.path.getsize(dev_path) if os.path.exists(dev_path) else 0} bytes)")
print(f"  test.txt:  {'✓' if os.path.exists(test_path) else '✗'} ({os.path.getsize(test_path) if os.path.exists(test_path) else 0} bytes)")

# Data Utils

In [ ]:
def read_data_file(data_path, verbose=False):
    """Read dataset file with format: [S][O][A][P][L]: [sentence] ===> ([P] ... [A] ... [S] ... [O] ... [L] ...)
    
    Example formats supported:
    - [P][A][S][O][L]: sentence ===> output
    - [S][O][A][P][L]: sentence ===> output
    - sentence ===> output
    """
    sents, labels = [], []
    
    try:
        with open(data_path, 'r', encoding='UTF-8') as fp:
            for line in fp:
                line = line.rstrip("\n").strip()
                if not line or '===>' not in line:
                    continue
                
                try:
                    # Split by ===> to get input and output parts
                    parts = line.split('===>')
                    if len(parts) != 2:
                        if verbose:
                            print(f"Skipping invalid line: {line}")
                        continue
                    
                    input_part = parts[0].strip()
                    output_part = parts[1].strip()
                    
                    # Remove various prefix formats: [P][A][S][O][L]:, [S][O][A][P][L]:, etc.
                    input_part = re.sub(r'^\[\w\]\[\w\]\[\w\]\[\w\]\[\w\]:\s*', '', input_part)
                    
                    # Skip if input or output is empty
                    if not input_part or not output_part:
                        if verbose:
                            print(f"Skipping empty input/output: {line}")
                        continue
                    
                    # Handle multiple labels separated by ;
                    for label in output_part.split(';'):
                        sents.append(input_part)
                        labels.append(label.strip())
                except Exception as e:
                    if verbose:
                        print(f"Error processing line: {line}. Error: {e}")
    except FileNotFoundError:
        print(f"File not found: {data_path}")
    except Exception as e:
        print(f"Error reading file: {data_path}. Error: {e}")
    
    return sents, labels

In [ ]:
# ========== DEBUG: Test data reading ==========
print("Testing data reading...")

# Test on a small sample
# Set verbose=False to reduce logs
test_inputs, test_labels = read_data_file(os.path.join(data_dir, 'train.txt'), verbose=False)

print(f"\n✓ Successfully read {len(test_inputs)} training samples")

if len(test_inputs) > 0:
    print(f"\n📝 Sample 1:")
    print(f"  Input:  {test_inputs[0]}")
    print(f"  Label:  {test_labels[0]}")
    
    if len(test_inputs) > 1:
        print(f"\n📝 Sample 2:")
        print(f"  Input:  {test_inputs[1]}")
        print(f"  Label:  {test_labels[1]}")
else:
    print("\n❌ ERROR: No samples read! Check file format.")
    print("Expected format: [P][A][S][O][L]: [sentence] ===> ([P] ... [A] ... [S] ... [O] ... [L] ...)")

In [ ]:
class CausalLMDataset(Dataset):
    """Dataset for Causal Language Models (Phi, Qwen)
    
    For causal LM, we combine input and output into a single sequence:
    Format: "Input: [sentence] Output: [label]"
    """
    def __init__(self, tokenizer, inputs=None, targets=None, max_len=256):
        self.tokenizer = tokenizer
        self.inputs = inputs or []
        self.targets = targets or []
        self.max_len = max_len
        self.encoded_data = self.encode(self.inputs, self.targets)

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        return self.encoded_data[idx]

    def encode(self, inputs=[], targets=[]):
        """Encode input-output pairs for causal LM
        
        For training: "Input: sentence\nOutput: structured_output"
        For test: "Input: sentence\nOutput: " (empty for generation)
        """
        encoded_data = []
        
        for i in range(len(inputs)):
            input_text = ' '.join(inputs[i]) if isinstance(inputs[i], list) else inputs[i]
            target_text = ' '.join(targets[i]) if isinstance(targets[i], list) else targets[i]
            
            # Handle both training (non-empty) and test (empty) data
            combined_text = f"Input: {input_text}\nOutput: {target_text}"
            
            # Tokenize combined text
            encoded = self.tokenizer(
                combined_text,
                max_length=self.max_len,
                padding='max_length',
                truncation=True,
                return_tensors="pt"
            )
            
            input_ids = encoded['input_ids'].squeeze()
            attention_mask = encoded['attention_mask'].squeeze()
            
            # Create labels: -100 for input part, token_ids for output part
            # For test data with empty output, mark input as -100
            input_part = f"Input: {input_text}\nOutput:"
            input_encoded = self.tokenizer(input_part, return_tensors="pt")
            input_ids_len = input_encoded['input_ids'].shape[-1]
            
            labels = input_ids.clone()
            # Only compute loss on output tokens
            labels[:input_ids_len] = -100
            
            # Handle scalar tensor shape
            if input_ids.dim() == 0:
                input_ids = input_ids.unsqueeze(0)
            if attention_mask.dim() == 0:
                attention_mask = attention_mask.unsqueeze(0)
            if labels.dim() == 0:
                labels = labels.unsqueeze(0)
            
            encoded_data.append({
                'input_ids': input_ids,
                'attention_mask': attention_mask,
                'labels': labels
            })
        
        return encoded_data

def get_dataset(file_path, tokenizer, max_len=256):
    """Load dataset from file"""
    inputs, targets = read_data_file(file_path)
    dataset = CausalLMDataset(tokenizer, inputs=inputs, targets=targets, max_len=max_len)
    return dataset

# Special Tokens

In [ ]:
SPECIAL_TOKENS = ['[S]', '[O]', '[A]', '[P]', '[L]', '[UNK]', '(', ')', ';', ':', 'Better', 'Worse', 'Equal', 'Different']

def prepare_constrained_vocab(name, data_dir):
    """Prepare constrained vocabulary from dataset"""
    inputs, _ = read_data_file(os.path.join(data_dir, f"{name}.txt"))
    constrained_vocab = set(" ".join(inputs).split())
    constrained_vocab.update(SPECIAL_TOKENS)
    return list(SPECIAL_TOKENS)

# Inference Function for Causal LM

In [ ]:
def infer(dataset, model, tokenizer, batch_size, name="eval", verbose=False):
    """Inference for causal LM - generates predictions"""
    data_loader = DataLoader(dataset, batch_size=batch_size, num_workers=0)
    
    inputs, outputs, targets = [], [], []
    average_loss = 0
    
    model.eval()
    
    with torch.no_grad():
        if name == "eval":
            # Evaluation: compute loss
            total_loss = 0
            num_batches = len(data_loader)
            
            for batch in tqdm(data_loader, desc="Evaluating", disable=not verbose):
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)
                
                outputs_batch = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels
                )
                loss = outputs_batch.loss
                total_loss += loss.item()
            
            average_loss = total_loss / num_batches
        else:
            # Inference: generate outputs
            for batch in tqdm(data_loader, desc=f"Inferencing ({name})", disable=not verbose):
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                
                # Generate
                generated_ids = model.generate(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    max_length=max_seq_length,
                    num_beams=1,
                    early_stopping=True,
                    do_sample=False
                )
                
                # Decode
                for i, gen_ids in enumerate(generated_ids):
                    text = tokenizer.decode(gen_ids, skip_special_tokens=False)
                    input_text = tokenizer.decode(input_ids[i], skip_special_tokens=True)
                    
                    # Extract output part after "Output:"
                    if "Output:" in text:
                        output_text = text.split("Output:")[-1].strip()
                    else:
                        output_text = text
                    
                    inputs.append(input_text)
                    outputs.append(output_text)
    
    # Save outputs
    with open(os.path.join(inference_dir, f"{name}_output.txt"), "w", encoding="utf-8") as f:
        for i, o in enumerate(outputs):
            f.write(f"{inputs[i]} ===> {o}\n")
    
    return average_loss, inputs, outputs, targets

# Evaluation Metrics

In [ ]:
import copy
from sklearn.metrics import f1_score, precision_recall_fscore_support

def extract_elements(input_string):
    """Extract structured elements from prediction"""
    input_list = input_string.split(';')
    pattern = re.compile(r'<sub>(.*?)<obj>(.*?)<asp>(.*?)<pred>(.*?)<lab>(.*?)$')
    result = []
    for i in input_list:
        i = i.strip()
        match = re.match(pattern, i[1:-1].strip())
        
        if match:
            items = match.groups()
            new_items = [item.strip() for item in items]
            result.append(new_items)
        else:
            result.append(None)
    
    return result

def compute_metrics(predicted_list, gold_list):
    """Compute precision, recall, F1 scores"""
    predicted_positions = list(map(list, zip(*predicted_list)))
    gold_positions = list(map(list, zip(*gold_list)))

    precision_scores = []
    recall_scores = []
    micro_f1_scores = []
    macro_f1_scores = []
    f1_scores = []

    for predicted, gold in zip(predicted_positions, gold_positions):
        micro_f1 = f1_score(predicted, gold, average='micro')
        micro_f1_scores.append(micro_f1)

        macro_f1 = f1_score(predicted, gold, average='macro')
        macro_f1_scores.append(macro_f1)

        p, r, f1, _ = precision_recall_fscore_support(predicted, gold, average=None)
        f1_scores.append(f1[0])
        precision_scores.append(p[0])
        recall_scores.append(r[0])

    return precision_scores, recall_scores, micro_f1_scores, macro_f1_scores, f1_scores

def eval(pred_tups, gold_tups, verbose="quiet", elem_dict=None):
    """Evaluate predictions against gold labels"""
    assert len(pred_tups) == len(gold_tups)

    elem_dict = elem_dict or []
    all_labels, all_predictions, error_preds = [], [], []
    
    for index in range(len(gold_tups)):
        predict_list = extract_elements(pred_tups[index])
        gold_list = extract_elements(gold_tups[index])
        
        if len(gold_list) > len(predict_list):
            error_preds.append(f"{index} Incomplete Prediction: {gold_tups[index]} ===> {pred_tups[index]}")
        
        for i in range(len(predict_list)):
            if i >= len(gold_list):
                error_preds.append(f"{index} Abundant Prediction: {pred_tups[index]}")
            elif predict_list[i] is None or gold_list[i] is None or len(gold_list[i]) != len(predict_list[i]) or len(predict_list[i]) != 5:
                error_preds.append(f"{index}: {gold_tups[index]} ===> {pred_tups[index]}")
            else:
                all_labels.append(gold_list[i])
                all_predictions.append(predict_list[i])

    precision_scores, recall_scores, micro_f1, macro_f1, f1_scores = compute_metrics(all_predictions, all_labels)

    scores_dict = {}
    for i, elem in enumerate(elem_dict):
        scores_dict[elem] = {
            "P": precision_scores[i],
            "R": recall_scores[i],
            "F1": f1_scores[i],
            "Macro-F1": macro_f1[i],
            "Micro-F1": micro_f1[i]
        }

    with open(os.path.join(inference_dir, 'error_prediction.txt'), 'w', encoding='utf-8') as fout:
        for error in error_preds:
            fout.write(f"{error}\n")

    print(f"Errors: {len(error_preds)}")
    if verbose != "quiet":
        print(f"Evaluation Result: {scores_dict}")

    return scores_dict

# Training Function for Causal LM

In [ ]:
from torch.cuda.amp import GradScaler

def train(model, tokenizer, train_data, val_data, epochs, lr, train_batch_size, eval_batch_size, acc_step=None, save_model=False, save_last=False, elem_dict=None):
    """Training function for Causal LM with memory optimization"""
    print("#" * 20 + " BEGIN TRAINING " + "#" * 20)
    
    # Enable gradient checkpointing to save memory
    if use_gradient_checkpointing:
        model.gradient_checkpointing_enable()
        print("✓ Gradient checkpointing enabled")
    
    # Set model to training mode
    model.train()
    
    no_decay = ["bias", "LayerNorm.weight"]
    optimizer_grouped_parameters = [
        {
            "params": [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
            "weight_decay": weight_decay,
        },
        {
            "params": [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],
            "weight_decay": 0.0,
        },
    ]
    optimizer = AdamW(optimizer_grouped_parameters, lr=lr, eps=adam_epsilon)

    if acc_step is None:
        acc_step = gradient_accumulation_steps
    
    train_loader = DataLoader(train_data, batch_size=train_batch_size, drop_last=False, shuffle=True, num_workers=0)
    
    # Avoid division by zero
    num_training_steps = max(1, (len(train_loader.dataset) // (train_batch_size * max(1, 1))) // acc_step * int(epochs))
    
    t_total = num_training_steps
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(num_warmup_steps), num_training_steps=int(t_total))
    train_iterator = trange(int(epochs), dynamic_ncols=True, desc="Epoch")
    
    train_losses, eval_losses = [], []
    
    # Initialize gradient scaler for mixed precision
    scaler = GradScaler(enabled=use_mixed_precision)
    
    for n_epoch, _ in enumerate(train_iterator):
        epoch_train_loss = 0.0
        epoch_iterator = tqdm(train_loader, dynamic_ncols=True, desc="Iteration", disable=False)
        step_count = 0

        for step, batch in enumerate(epoch_iterator):
            model.train()

            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            # Forward pass with gradient scaling
            with torch.cuda.amp.autocast(enabled=use_mixed_precision):
                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels
                )
                loss = outputs.loss
                loss = loss / acc_step  # Scale loss by accumulation steps
            
            # Backward pass
            scaler.scale(loss).backward()
            epoch_train_loss += loss.item() * acc_step
            step_count += 1

            # Optimizer step
            if (step + 1) % acc_step == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                model.zero_grad()
                
                # Clear cache to free memory
                torch.cuda.empty_cache()
        
        # Evaluation
        eval_loss, _, _, _ = infer(val_data, model, tokenizer, batch_size=eval_batch_size, name="eval", verbose=False)
        eval_losses.append(eval_loss)
        
        train_losses.append(epoch_train_loss / max(1, step_count))
        
        print(f"Epoch {n_epoch} - Train Loss: {epoch_train_loss / max(1, step_count):.5f} | Eval Loss: {eval_loss:.5f}")
        
        # Clear GPU memory
        torch.cuda.empty_cache()

    if save_last:
        save_dir = os.path.join(result_dir, f"{MODEL_NAME}-final-model")
        if not os.path.exists(save_dir):
            os.makedirs(save_dir)

        model.save_pretrained(save_dir)
        tokenizer.save_pretrained(save_dir)
        print(f"Model saved to {save_dir}")
    
    # Save losses
    with open(os.path.join(inference_dir, "train_losses.txt"), 'w') as f:
        for loss in train_losses:
            f.write(f"{loss}\n")
    
    with open(os.path.join(inference_dir, "eval_losses.txt"), 'w') as f:
        for loss in eval_losses:
            f.write(f"{loss}\n")

    print("#" * 20 + " FINISH TRAINING " + "#" * 20)

# Main Training Pipeline

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

test_file = 'test.txt'
train_file = 'train.txt'
dev_file = 'dev.txt'

def main(do_train, do_test, test_label=False):
    print(f"\n{'='*60}")
    print(f"Training Small LLM: {MODEL_NAME}")
    print(f"Model: {model_checkpoint}")
    print(f"{'='*60}\n")
    
    if do_train:
        # Clear GPU memory before starting
        torch.cuda.empty_cache()
        
        # Load tokenizer and model
        print("Loading tokenizer...")
        tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, use_fast=False, trust_remote_code=True)
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.add_tokens(SPECIAL_TOKENS)
        print(f"Tokenizer loaded. Vocab size: {len(tokenizer)}")
        
        # Load data
        train_inputs, train_labels = read_data_file(os.path.join(data_dir, train_file))
        eval_inputs, eval_labels = read_data_file(os.path.join(data_dir, dev_file))
        
        print(f"Train samples: {len(train_inputs)}")
        print(f"Eval samples: {len(eval_inputs)}")
        
        # Create datasets
        print("Creating datasets...")
        train_data = get_dataset(os.path.join(data_dir, train_file), tokenizer=tokenizer, max_len=max_seq_length)
        eval_data = get_dataset(os.path.join(data_dir, dev_file), tokenizer=tokenizer, max_len=max_seq_length)
        
        # Load model with memory optimizations
        print("Loading model with memory optimizations...")
        torch.cuda.empty_cache()
        
        model = AutoModelForCausalLM.from_pretrained(
            model_checkpoint, 
            trust_remote_code=True,
            torch_dtype=torch.float16 if use_mixed_precision else torch.float32,
            device_map="auto"
        )
        model.resize_token_embeddings(len(tokenizer))
        
        if not use_mixed_precision:
            model.to(device)
        
        print(f"Model loaded: {model_checkpoint}")
        print(f"Total parameters: {model.num_parameters():,}")
        print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}\n")
        
        # Show sample
        print("Sample from dataset:")
        for i in range(min(2, len(train_data))):
            sample = train_data[i]
            print(f"  Input: {tokenizer.decode(sample['input_ids'], skip_special_tokens=True)[:100]}...")
        print()
        
        # Train
        train(
            model, tokenizer, train_data, val_data=eval_data,
            epochs=num_train_epochs, lr=lr,
            train_batch_size=train_batch_size,
            eval_batch_size=eval_batch_size,
            save_model=True, save_last=True,
            elem_dict=elem_dict
        )
        
        # Clear memory after training
        torch.cuda.empty_cache()
    
    if do_test:
        print("\n" + "*" * 60 + " TESTING " + "*" * 60)
        
        # Load best model
        model_dir = os.path.join(result_dir, f"{MODEL_NAME}-final-model")
        
        if os.path.exists(model_dir):
            torch.cuda.empty_cache()
            
            model = AutoModelForCausalLM.from_pretrained(
                model_dir, 
                trust_remote_code=True,
                torch_dtype=torch.float16 if use_mixed_precision else torch.float32,
                device_map="auto"
            )
            tokenizer = AutoTokenizer.from_pretrained(model_dir, use_fast=False, trust_remote_code=True)
            
            if not use_mixed_precision:
                model.to(device)
            
            # Load test data
            test_data = get_dataset(os.path.join(data_dir, test_file), tokenizer=tokenizer, max_len=max_seq_length)
            
            print(f"Test samples: {len(test_data)}")
            
            # Inference
            _, sents, predictions, _ = infer(test_data, model, tokenizer, batch_size=eval_batch_size, name="test")
            
            print(f"\nSample predictions:")
            for i in range(min(3, len(predictions))):
                print(f"  Input: {sents[i][:80]}...")
                print(f"  Pred:  {predictions[i][:80]}...\n")
        else:
            print(f"Model not found at {model_dir}")
            print("Please run training first (do_train=True)")

# Run Training & Testing

In [ ]:
# Run training and testing
MODEL_NAME = 'qwen-0.5b'
main(do_train=True, do_test=True, test_label=False)

# Compare Multiple Models

In [ ]:
"""Run this cell to train and test multiple models
This will compare performance of different small LLMs
"""

# Uncomment and modify to test multiple models
# for model_name in ['phi-2', 'qwen-0.5b', 'qwen-1b']:
#     MODEL_NAME = model_name
#     model_checkpoint = AVAILABLE_MODELS[MODEL_NAME]
#     result_dir = f"/kaggle/working/result/model-{MODEL_NAME}"
#     inference_dir = f"/kaggle/working/result/inference-{MODEL_NAME}"
#     
#     os.makedirs(result_dir, exist_ok=True)
#     os.makedirs(inference_dir, exist_ok=True)
#     
#     print(f"\n\n{'='*80}")
#     print(f"Training model: {MODEL_NAME}")
#     print(f"{'='*80}\n")
#     
#     main(do_train=True, do_test=True, test_label=False)